# Telecome Churn Predixtion

## 1. Load data

In [1]:
import pandas as pd
import os

file_path = "data/telecom_customer_churn.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"✅ Dataset loaded successfully!")
    df.head()
else:
    print(f"❌ File not found at: {file_path}")

✅ Dataset loaded successfully!


## 2- Dataset Overview

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 38 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   str    
 1   Gender                             7043 non-null   str    
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   str    
 4   Number of Dependents               7043 non-null   int64  
 5   City                               7043 non-null   str    
 6   Zip Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Number of Referrals                7043 non-null   int64  
 10  Tenure in Months                   7043 non-null   int64  
 11  Offer                              3166 non-null   str    
 12  Pho

In [3]:
# === DUPLICATION CHECK ===

print("="*50)
print("DUPLICATION ANALYSIS")
print("="*50)

# Check for duplicate Customer IDs (should be unique identifier)
duplicate_customer_ids = df['Customer ID'].duplicated().sum()
print(f"Duplicate Customer IDs: {duplicate_customer_ids}")

# Check for null Customer IDs
null_customer_ids = df['Customer ID'].isnull().sum()
print(f"Null Customer IDs: {null_customer_ids}")

# Summary statistics
print(f"\nTotal rows: {len(df)}")
print(f"Unique Customer IDs: {df['Customer ID'].nunique()}")

# Show duplicates if any exist
if duplicate_customer_ids > 0:
    print("\n⚠️ Duplicate Customer IDs found:")
    duplicate_ids = df[df['Customer ID'].duplicated(keep=False)]['Customer ID'].unique()
    print(f"Duplicate Customer IDs: {duplicate_ids}")
    print("\nRows with duplicate IDs:")
    display(df[df['Customer ID'].isin(duplicate_ids)].sort_values('Customer ID'))
else:
    print("\n✅ All Customer IDs are unique! No duplicates found.")

# Recommendation
print("\n" + "="*50)
print("RECOMMENDATION")
print("="*50)

if duplicate_customer_ids == 0 and null_customer_ids == 0:
    print("✅ Dataset is clean. Proceed to data splitting.")
else:
    print("⚠️ Duplicate or null Customer IDs detected. Need to resolve before splitting.")

DUPLICATION ANALYSIS
Duplicate Customer IDs: 0
Null Customer IDs: 0

Total rows: 7043
Unique Customer IDs: 7043

✅ All Customer IDs are unique! No duplicates found.

RECOMMENDATION
✅ Dataset is clean. Proceed to data splitting.


In [4]:
# === MISSING VALUES ANALYSIS ===

print("="*50)
print("MISSING VALUES ANALYSIS")
print("="*50)

# 1. Overall missing values summary
missing_summary = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_summary,
    'Missing Percentage': missing_percentage
})

# Filter to show only columns with missing values
missing_columns = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_columns) > 0:
    print(f"Columns with missing values: {len(missing_columns)}")
    print("\nDetailed missing values:")
    display(missing_columns)
else:
    print("✅ No missing values found in any column!")

# 2. Check if any row has all missing values
rows_all_missing = df.isnull().all(axis=1).sum()
print(f"\nRows with all columns missing: {rows_all_missing}")

MISSING VALUES ANALYSIS
Columns with missing values: 15

Detailed missing values:


,Missing Count,Missing Percentage
Churn Category,5174,73.463013
Churn Reason,5174,73.463013
Offer,3877,55.047565
Online Security,1526,21.666903
Online Backup,1526,21.666903
Avg Monthly GB Download,1526,21.666903
Internet Type,1526,21.666903
Streaming Movies,1526,21.666903
Streaming TV,1526,21.666903
Device Protection Plan,1526,21.666903



Rows with all columns missing: 0


In [5]:
# === VALIDATE MISSING VALUES CONSISTENCY ===

print("="*50)
print("MISSING VALUES CONSISTENCY CHECK")
print("="*50)

# Phone services validation
no_phone = df[df['Phone Service'] == 'No']
print(f"\nPhone Service:")
print(f"  - Customers with 'No' Phone Service: {len(no_phone)}")
print(f"  - Missing in 'Avg Monthly Long Distance Charges': {no_phone['Avg Monthly Long Distance Charges'].isnull().sum()} (should be {len(no_phone)})")
print(f"  - Missing in 'Multiple Lines': {no_phone['Multiple Lines'].isnull().sum()} (should be {len(no_phone)})")

# Internet services validation
no_internet = df[df['Internet Service'] == 'No']
internet_cols = ['Internet Type', 'Avg Monthly GB Download', 'Online Security', 'Online Backup', 
                 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 
                 'Streaming Music', 'Unlimited Data']

print(f"\nInternet Service:")
print(f"  - Customers with 'No' Internet Service: {len(no_internet)}")
for col in internet_cols:
    missing = no_internet[col].isnull().sum()
    print(f"  - Missing in '{col}': {missing} (should be {len(no_internet)})")

# Quick conclusion
print("\n" + "="*50)
all_phone_match = (no_phone['Avg Monthly Long Distance Charges'].isnull().sum() == len(no_phone)) and (no_phone['Multiple Lines'].isnull().sum() == len(no_phone))
all_internet_match = all(no_internet[col].isnull().sum() == len(no_internet) for col in internet_cols)

if all_phone_match and all_internet_match:
    print("✅ All missing values perfectly match customers without the service.")
else:
    print("⚠️ Inconsistencies found - investigate further.")

MISSING VALUES CONSISTENCY CHECK

Phone Service:
  - Customers with 'No' Phone Service: 682
  - Missing in 'Avg Monthly Long Distance Charges': 682 (should be 682)
  - Missing in 'Multiple Lines': 682 (should be 682)

Internet Service:
  - Customers with 'No' Internet Service: 1526
  - Missing in 'Internet Type': 1526 (should be 1526)
  - Missing in 'Avg Monthly GB Download': 1526 (should be 1526)
  - Missing in 'Online Security': 1526 (should be 1526)
  - Missing in 'Online Backup': 1526 (should be 1526)
  - Missing in 'Device Protection Plan': 1526 (should be 1526)
  - Missing in 'Premium Tech Support': 1526 (should be 1526)
  - Missing in 'Streaming TV': 1526 (should be 1526)
  - Missing in 'Streaming Movies': 1526 (should be 1526)
  - Missing in 'Streaming Music': 1526 (should be 1526)
  - Missing in 'Unlimited Data': 1526 (should be 1526)

✅ All missing values perfectly match customers without the service.


In [6]:
# === DISTRIBUTION CHECK ===

print("="*50)
print("DISTRIBUTION ANALYSIS")
print("="*50)
df['Churn'] = (df['Customer Status'] == 'Churned').astype(int)

# 1. Target variable distribution
print("\n1. TARGET VARIABLE (Churn):")
print(df['Churn'].value_counts())
print(f"Churn rate: {df['Churn'].mean()*100:.2f}%")

# 2. Numerical columns distribution
print("\n2. NUMERICAL FEATURES:")
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    if col != 'Churn':  # Skip target
        print(f"\n{col}:")
        print(f"  - Min: {df[col].min()}")
        print(f"  - Max: {df[col].max()}")
        print(f"  - Mean: {df[col].mean():.2f}")
        print(f"  - Median: {df[col].median()}")
        print(f"  - Std: {df[col].std():.2f}")

# 3. Categorical columns distribution (top categories)
print("\n3. CATEGORICAL FEATURES (top 3 categories):")
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts().head(3))

DISTRIBUTION ANALYSIS

1. TARGET VARIABLE (Churn):
Churn
0    5174
1    1869
Name: count, dtype: int64
Churn rate: 26.54%

2. NUMERICAL FEATURES:

Age:
  - Min: 19
  - Max: 80
  - Mean: 46.51
  - Median: 46.0
  - Std: 16.75

Number of Dependents:
  - Min: 0
  - Max: 9
  - Mean: 0.47
  - Median: 0.0
  - Std: 0.96

Zip Code:
  - Min: 90001
  - Max: 96150
  - Mean: 93486.07
  - Median: 93518.0
  - Std: 1856.77

Latitude:
  - Min: 32.555828
  - Max: 41.962127
  - Mean: 36.20
  - Median: 36.205465
  - Std: 2.47

Longitude:
  - Min: -124.301372
  - Max: -114.192901
  - Mean: -119.76
  - Median: -119.595293
  - Std: 2.15

Number of Referrals:
  - Min: 0
  - Max: 11
  - Mean: 1.95
  - Median: 0.0
  - Std: 3.00

Tenure in Months:
  - Min: 1
  - Max: 72
  - Mean: 32.39
  - Median: 29.0
  - Std: 24.54

Avg Monthly Long Distance Charges:
  - Min: 1.01
  - Max: 49.99
  - Mean: 25.42
  - Median: 25.69
  - Std: 14.20

Avg Monthly GB Download:
  - Min: 2.0
  - Max: 85.0
  - Mean: 26.19
  - Median: 21.

C:\Users\Bahar\AppData\Local\Temp\ipykernel_13636\884297187.py:27: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns


## 3- Splitting Dataset

In [7]:
# === DROP LEAKAGE COLUMNS AND 'JOINED' CUSTOMERS ===

# Drop post-churn columns
df = df.drop(columns=['Churn Category', 'Churn Reason', 'Customer ID'])

# Remove 'Joined' customers
df = df[df['Customer Status'] != 'Joined']

# Remove city and zip-code and relying on longitude latitude for reigonal data
df = df.drop(columns=["Zip Code", "City"])

# Count rows with negative Monthly Charge and Remove them
negative_count = (df['Monthly Charge'] < 0).sum()

df = df[df['Monthly Charge'] >= 0]

# Create binary target
df = df.drop(columns=['Customer Status'])

print(f"Remaining customers: {len(df)}")
print(f"Columns remaining: {len(df.columns)}")
df.columns

Remaining customers: 6475
Columns remaining: 33


Index(['Gender', 'Age', 'Married', 'Number of Dependents', 'Latitude',
       'Longitude', 'Number of Referrals', 'Tenure in Months', 'Offer',
       'Phone Service', 'Avg Monthly Long Distance Charges', 'Multiple Lines',
       'Internet Service', 'Internet Type', 'Avg Monthly GB Download',
       'Online Security', 'Online Backup', 'Device Protection Plan',
       'Premium Tech Support', 'Streaming TV', 'Streaming Movies',
       'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing',
       'Payment Method', 'Monthly Charge', 'Total Charges', 'Total Refunds',
       'Total Extra Data Charges', 'Total Long Distance Charges',
       'Total Revenue', 'Churn'],
      dtype='str')

In [8]:
# === HANDLE MISSING VALUES ===

print("="*50)
print("HANDLING MISSING VALUES")
print("="*50)

# 1. Phone services
df['Avg Monthly Long Distance Charges'] = df['Avg Monthly Long Distance Charges'].fillna(0)
df['Multiple Lines'] = df['Multiple Lines'].fillna('No Phone Service')

# 2. Internet services (categorical)
internet_categorical = [
    'Internet Type', 'Online Security', 'Online Backup', 
    'Device Protection Plan', 'Premium Tech Support',
    'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data'
]
for col in internet_categorical:
    df[col] = df[col].fillna('No Internet Service')

# 3. Internet service (numeric)
df['Avg Monthly GB Download'] = df['Avg Monthly GB Download'].fillna(0)

# 4. Offer
df['Offer'] = df['Offer'].fillna('No Offer')

# Verify
remaining_missing = df.isnull().sum().sum()
print(f"Missing values before: 8,536")
print(f"Missing values after: {remaining_missing}")

if remaining_missing == 0:
    print("✅ All missing values handled successfully!")
else:
    print(f"⚠️ Still have {remaining_missing} missing values")
    print("\nColumns with remaining missing values:")
    print(df.isnull().sum()[df.isnull().sum() > 0])

HANDLING MISSING VALUES
Missing values before: 8,536
Missing values after: 0
✅ All missing values handled successfully!


In [10]:
# === DATA SPLITTING ===

from sklearn.model_selection import train_test_split

print("="*50)
print("DATA SPLITTING")
print("="*50)

# Separate features and target
X = df.drop(columns=['Churn'])
y = df['Churn']

# Split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print("Training set label distribution:")
print(y_train.value_counts())
print(f"Churn rate: {y_train.mean()*100:.2f}%")
print("\nTest set label distribution:")
print(y_test.value_counts())
print(f"Churn rate: {y_test.mean()*100:.2f}%")

print(f"\nTraining set: {len(X_train)} rows")
print(f"Test set: {len(X_test)} rows")
print(f"\n✅ Stratified split completed!")

# Save to data directory
import os
os.makedirs('data', exist_ok=True)

X_train.to_csv('data/X_train.csv', index=False)
X_test.to_csv('data/X_test.csv', index=False)
y_train.to_csv('data/y_train.csv', index=False)
y_test.to_csv('data/y_test.csv', index=False)

print("\n✅ Data saved to data/ directory")

DATA SPLITTING
Training set label distribution:
Churn
0    3709
1    1471
Name: count, dtype: int64
Churn rate: 28.40%

Test set label distribution:
Churn
0    927
1    368
Name: count, dtype: int64
Churn rate: 28.42%

Training set: 5180 rows
Test set: 1295 rows

✅ Stratified split completed!

✅ Data saved to data/ directory
